# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [16]:

df = pd.read_csv(
    "data/AviationData.csv",
    encoding="latin1",
    low_memory=False
)

print("Dataset shape:", df.shape)

display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False))

Dataset shape: (88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980



Column names:
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date']

Data types:


Event.Id                      str
Investigation.Type            str
Accident.Number               str
Event.Date                    str
Location                      str
Country                       str
Latitude                      str
Longitude                     str
Airport.Code                  str
Airport.Name                  str
Injury.Severity               str
Aircraft.damage               str
Aircraft.Category             str
Registration.Number           str
Make                          str
Model                         str
Amateur.Built                 str
Number.of.Engines         float64
Engine.Type                   str
FAR.Description               str
Schedule                      str
Purpose.of.flight             str
Air.carrier                   str
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
Weather.Condition             str
Broad.phase.of.flight         str
Report.Status 


Missing values by column:


Schedule                  76307
Air.carrier               72241
FAR.Description           56866
Aircraft.Category         56602
Longitude                 54516
Latitude                  54507
Airport.Code              38757
Airport.Name              36185
Broad.phase.of.flight     27165
Publication.Date          13771
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Engine.Type                7096
Report.Status              6384
Purpose.of.flight          6192
Number.of.Engines          6084
Total.Uninjured            5912
Weather.Condition          4492
Aircraft.damage            3194
Registration.Number        1382
Injury.Severity            1000
Country                     226
Amateur.Built               102
Model                        92
Make                         63
Location                     52
Accident.Number               0
Investigation.Type            0
Event.Id                      0
Event.Date                    0
dtype: i

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [17]:

eligibility_columns = [
    "Event.Date",
    "Aircraft.Category",
    "Amateur.Built"
]

for column in eligibility_columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False).head(20))

print("\nOriginal dataset shape:", df.shape)


df["Event.Date"] = pd.to_datetime(
    df["Event.Date"],
    errors="coerce"
)

start_date = pd.Timestamp("1983-01-01")
end_date = pd.Timestamp("2023-12-31")


invalid_dates = df["Event.Date"].isna().sum()
print("\nInvalid or missing event dates:", invalid_dates)


eligible_df = df.loc[
    df["Event.Date"].between(start_date, end_date)
    & df["Aircraft.Category"].str.strip().str.casefold().eq("airplane")
    & df["Amateur.Built"].str.strip().str.casefold().eq("no")
].copy()

print("\nEligible dataset shape:", eligible_df.shape)

print("\nDate range after filtering:")
print(eligible_df["Event.Date"].min())
print(eligible_df["Event.Date"].max())

print("\nAircraft categories retained:")
print(eligible_df["Aircraft.Category"].value_counts())

print("\nAmateur-built status retained:")
print(eligible_df["Amateur.Built"].value_counts())


--- Event.Date ---
Event.Date
1982-05-16    25
1984-06-30    25
2000-07-08    25
1983-06-05    24
1983-08-05    24
1984-08-25    24
1986-05-17    24
1982-05-01    23
1982-07-09    23
1982-10-03    23
1983-05-28    23
1988-08-07    23
2001-06-16    23
2001-07-21    23
1982-04-25    22
1983-05-30    22
1983-08-06    22
1983-09-10    22
1983-10-01    22
1984-05-27    22
Name: count, dtype: int64

--- Aircraft.Category ---
Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

--- Amateur.Built ---
Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int6

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [18]:
# Work with the eligible aircraft records
# without modifying the original DataFrame.
injury_df = eligible_df.copy()

# Injury columns to clean
injury_columns = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

# Convert injury counts to numeric.
# Invalid values become NaN.
for column in injury_columns:
    injury_df[column] = pd.to_numeric(
        injury_df[column],
        errors="coerce"
    )

    # Negative injury counts are invalid.
    injury_df.loc[
        injury_df[column] < 0,
        column
    ] = np.nan

# Require complete injury information for the metrics.
# Missing values are not assumed to represent zero injuries.
complete_counts = injury_df[injury_columns].notna().all(axis=1)

# Estimate total occupants only for records with
# complete and valid injury counts.
injury_df["Estimated.Occupants"] = np.nan

injury_df.loc[complete_counts, "Estimated.Occupants"] = (
    injury_df.loc[complete_counts, injury_columns].sum(axis=1)
)

# Calculate fatal and serious injuries.
injury_df["Fatal.Serious.Injuries"] = (
    injury_df["Total.Fatal.Injuries"]
    + injury_df["Total.Serious.Injuries"]
)

# Flag records with valid occupant counts and plausible totals.
valid_counts = (
    complete_counts
    & injury_df["Estimated.Occupants"].gt(0)
    & injury_df["Fatal.Serious.Injuries"].le(
        injury_df["Estimated.Occupants"]
    )
)

# Calculate the fraction of occupants who were
# fatally or seriously injured.
injury_df["Fatal.Serious.Injury.Fraction"] = np.nan

injury_df.loc[valid_counts, "Fatal.Serious.Injury.Fraction"] = (
    injury_df.loc[valid_counts, "Fatal.Serious.Injuries"]
    / injury_df.loc[valid_counts, "Estimated.Occupants"]
)

# Inspect the derived metrics.
print("Eligible records:", len(injury_df))
print("Records with complete injury counts:", complete_counts.sum())
print("Records with valid injury metrics:", valid_counts.sum())
print(
    "Records without valid injury metrics:",
    (~valid_counts).sum()
)

display(
    injury_df[
        injury_columns
        + [
            "Estimated.Occupants",
            "Fatal.Serious.Injuries",
            "Fatal.Serious.Injury.Fraction"
        ]
    ].head(10)
)

print("\nSummary statistics for injury fraction:")
display(
    injury_df["Fatal.Serious.Injury.Fraction"].describe()
)

Eligible records: 21447
Records with complete injury counts: 18419
Records with valid injury metrics: 17562
Records without valid injury metrics: 3885


,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Estimated.Occupants,Fatal.Serious.Injuries,Fatal.Serious.Injury.Fraction
4149,NaN,NaN,NaN,588.0,NaN,NaN,NaN
4150,NaN,NaN,NaN,588.0,NaN,NaN,NaN
4171,1.0,1.0,NaN,NaN,NaN,2.0,NaN
4285,1.0,NaN,NaN,4.0,NaN,NaN,NaN
5957,NaN,NaN,NaN,289.0,NaN,NaN,NaN
5960,11.0,2.0,NaN,13.0,NaN,13.0,NaN
6669,1.0,NaN,NaN,29.0,NaN,NaN,NaN
6760,NaN,NaN,NaN,100.0,NaN,NaN,NaN
6806,NaN,NaN,NaN,1.0,NaN,NaN,NaN
7084,NaN,NaN,NaN,1.0,NaN,NaN,NaN



Summary statistics for injury fraction:


count    17562.000000
mean         0.312429
std          0.442805
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000
Name: Fatal.Serious.Injury.Fraction, dtype: float64

### Aircraft Damage Cleaning and Derived Metric

The `Aircraft.damage` column records the extent of damage sustained by an aircraft. To make this information suitable for analysis, we standardize the text values by removing unnecessary whitespace and normalizing capitalization.

Missing or unrecognized damage values are retained as missing or classified as unknown rather than being assumed to represent a specific damage outcome.

A new binary column, `Aircraft.Destroyed`, is constructed from the cleaned damage information. It is `1` when the aircraft was recorded as destroyed, `0` when it was recorded as substantially or slightly damaged, and missing when the damage status is unavailable or unrecognized.

In [19]:
# Inspect the original aircraft damage values
print("Original aircraft damage values:")
display(injury_df["Aircraft.damage"].value_counts(dropna=False))

# Standardize the damage descriptions
injury_df["Aircraft.damage"] = (
    injury_df["Aircraft.damage"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

# Convert empty strings to missing values
injury_df["Aircraft.damage"] = injury_df["Aircraft.damage"].replace(
    "", pd.NA
)

# Create a binary indicator: 1 = destroyed, 0 = not destroyed
damage_mapping = {
    "destroyed": 1,
    "substantial": 0,
    "minor": 0
}

injury_df["Aircraft.Destroyed"] = (
    injury_df["Aircraft.damage"].map(damage_mapping)
)

# Review the cleaned values and derived column
print("\nCleaned aircraft damage values:")
display(injury_df["Aircraft.damage"].value_counts(dropna=False))

print("\nDestroyed status counts (1 = destroyed, 0 = not destroyed):")
display(injury_df["Aircraft.Destroyed"].value_counts(dropna=False))

print("\nRecords with unrecognized damage descriptions:")
display(
    injury_df.loc[
        injury_df["Aircraft.damage"].notna()
        & ~injury_df["Aircraft.damage"].isin(damage_mapping),
        "Aircraft.damage"
    ].value_counts()
)

Original aircraft damage values:


Aircraft.damage
Substantial    16990
Destroyed       2316
NaN             1227
Minor            817
Unknown           97
Name: count, dtype: int64


Cleaned aircraft damage values:


Aircraft.damage
substantial    16990
destroyed       2316
<NA>            1227
minor            817
unknown           97
Name: count, dtype: Int64


Destroyed status counts (1 = destroyed, 0 = not destroyed):


Aircraft.Destroyed
0.0    17807
1.0     2316
NaN     1324
Name: count, dtype: int64


Records with unrecognized damage descriptions:


Aircraft.damage
unknown    97
Name: count, dtype: Int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [20]:
# Work on a separate copy so other analyses retain all eligible records
make_df = injury_df.copy()

# Inspect missing and blank make values
print("Missing Make values before cleaning:", make_df["Make"].isna().sum())

blank_make = (
    make_df["Make"].astype("string").str.strip().eq("").fillna(False)
)
print("Blank Make values before cleaning:", blank_make.sum())

print("\nTop 20 original Make values:")
display(make_df["Make"].value_counts(dropna=False).head(20))

# Clean Make names: trim whitespace and standardize capitalization
make_df["Make"] = (
    make_df["Make"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Treat blank names as missing
make_df["Make"] = make_df["Make"].replace("", pd.NA)

print("\nMissing Make values after cleaning:", make_df["Make"].isna().sum())

# Exclude missing Make values from make-level analysis
make_df = make_df.dropna(subset=["Make"]).copy()

# Count eligible accident records for each Make
make_counts = make_df["Make"].value_counts()

# Keep only makes with at least 50 records
minimum_records = 50
qualifying_makes = make_counts[make_counts >= minimum_records].index

make_analysis_df = make_df[
    make_df["Make"].isin(qualifying_makes)
].copy()

# Report the filtering results
print("\nRecords after excluding missing Make values:", len(make_df))
print("Unique makes before threshold:", make_counts.size)
print("Minimum records per make:", minimum_records)
print("Makes retained:", len(qualifying_makes))
print("Records retained for make-level analysis:", len(make_analysis_df))
print("Records excluded by the threshold:", len(make_df) - len(make_analysis_df))

print("\nMost frequent qualifying makes:")
display(make_analysis_df["Make"].value_counts().head(20))

Missing Make values before cleaning: 3
Blank Make values before cleaning: 0

Top 20 original Make values:


Make
CESSNA                4867
PIPER                 2803
Cessna                2279
Piper                 1186
BOEING                1037
BEECH                 1018
Beech                  413
MOONEY                 238
Boeing                 227
CIRRUS DESIGN CORP     218
AIR TRACTOR INC        217
AIRBUS                 215
BELLANCA               158
AERONCA                149
MAULE                  144
Mooney                 125
EMBRAER                123
Air Tractor            117
LUSCOMBE                95
STINSON                 91
Name: count, dtype: int64


Missing Make values after cleaning: 3

Records after excluding missing Make values: 21444
Unique makes before threshold: 1087
Minimum records per make: 50
Makes retained: 36
Records retained for make-level analysis: 17892
Records excluded by the threshold: 3552

Most frequent qualifying makes:


Make
CESSNA                7146
PIPER                 3989
BEECH                 1431
BOEING                1264
MOONEY                 363
AIRBUS                 243
CIRRUS DESIGN CORP     220
BELLANCA               219
AIR TRACTOR INC        219
MAULE                  215
AIR TRACTOR            206
AERONCA                200
CHAMPION               158
EMBRAER                153
GRUMMAN                147
LUSCOMBE               141
CIRRUS                 137
STINSON                129
MCDONNELL DOUGLAS      108
NORTH AMERICAN         106
Name: count, dtype: Int64

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [21]:
# Create a separate DataFrame for model-level analysis
model_df = injury_df.copy()

# Check missing and blank values before cleaning
print("Missing Model values:", model_df["Model"].isna().sum())

blank_model = (
    model_df["Model"]
    .astype("string")
    .str.strip()
    .eq("")
    .fillna(False)
)
print("Blank Model values:", blank_model.sum())

print("Missing Make values:", model_df["Make"].isna().sum())

# Standardize Make and Model labels
model_df["Make"] = (
    model_df["Make"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

model_df["Model"] = (
    model_df["Model"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

# Exclude records with missing Make or Model
model_df = model_df.dropna(subset=["Make", "Model"]).copy()

print("\nRecords retained after removing missing Make/Model:", len(model_df))
print("Unique makes:", model_df["Make"].nunique())
print("Unique model labels:", model_df["Model"].nunique())

# Count accidents for each Make and Model combination
make_model_counts = (
    model_df.groupby(["Make", "Model"], dropna=False)
    .size()
    .reset_index(name="Accident.Count")
    .sort_values("Accident.Count", ascending=False)
)

print("\nTop 20 Make-Model combinations:")
display(make_model_counts.head(20))

# Check whether any model label appears under multiple makes
makes_per_model = model_df.groupby("Model")["Make"].nunique()

shared_model_labels = makes_per_model[makes_per_model > 1]

print(
    "\nModel labels associated with more than one make:",
    len(shared_model_labels)
)

print("\nExamples of model labels shared across makes:")
display(
    model_df[model_df["Model"].isin(shared_model_labels.index)]
    [["Make", "Model"]]
    .drop_duplicates()
    .sort_values(["Model", "Make"])
    .head(30)
)

# Create a unique aircraft-type identifier
model_df["Aircraft.Type"] = model_df["Make"] + " | " + model_df["Model"]

print("\nUnique aircraft types:", model_df["Aircraft.Type"].nunique())
print("Unique Make-Model combinations:", len(make_model_counts))

# Verify that each identifier maps to exactly one Make-Model combination
identifier_check = model_df.groupby("Aircraft.Type")[["Make", "Model"]].nunique()
print(
    "Identifiers mapping to multiple makes or models:",
    (identifier_check > 1).any(axis=1).sum()
)

Missing Model values: 18
Blank Model values: 0
Missing Make values: 3

Records retained after removing missing Make/Model: 21428
Unique makes: 1087
Unique model labels: 3467

Top 20 Make-Model combinations:


,Make,Model,Accident.Count
1297,CESSNA,172,769
861,BOEING,737,403
1287,CESSNA,152,316
1360,CESSNA,182,304
1333,CESSNA,172S,276
3544,PIPER,PA28,273
1328,CESSNA,172N,249
1345,CESSNA,180,213
1327,CESSNA,172M,180
1272,CESSNA,150,179



Model labels associated with more than one make: 553

Examples of model labels shared across makes:


,Make,Model
55185,AERO COMMANDER,100
67293,BEECH,100
40824,FOKKER,100
65551,HAWKER-BEECHCRAFT,100
84829,OMF,100
57169,STINSON,108
72035,UNIVERSAL STINSON,108
74609,STINSON,108 1
74269,UNIVERSAL STINSON,108 1
56409,STINSON,108-1



Unique aircraft types: 4355
Unique Make-Model combinations: 4355
Identifiers mapping to multiple makes or models: 0


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [22]:
columns_to_inspect = [
    "Engine.Type",
    "Weather.Condition",
    "Number.of.Engines",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]

for column in columns_to_inspect:
    print(f"\n{'=' * 60}")
    print(f"Column: {column}")
    print("Data type:", injury_df[column].dtype)
    print("Missing values:", injury_df[column].isna().sum())
    print("Unique values:", injury_df[column].nunique(dropna=True))
    print("Value counts:")

    display(injury_df[column].value_counts(dropna=False).head(30))


Column: Engine.Type
Data type: str
Missing values: 3956
Unique values: 9
Value counts:


Engine.Type
Reciprocating      15015
NaN                 3956
Turbo Prop          1251
Turbo Fan            929
Unknown              134
Turbo Jet            133
Geared Turbofan       12
Turbo Shaft           11
Electric               5
UNK                    1
Name: count, dtype: int64


Column: Weather.Condition
Data type: str
Missing values: 2979
Unique values: 4
Value counts:


Weather.Condition
VMC    17089
NaN     2979
IMC     1069
Unk      215
UNK       95
Name: count, dtype: int64


Column: Number.of.Engines
Data type: float64
Missing values: 2551
Unique values: 7
Value counts:


Number.of.Engines
1.0    15742
2.0     3010
NaN     2551
4.0       93
3.0       40
0.0        9
8.0        1
6.0        1
Name: count, dtype: int64


Column: Purpose.of.flight
Data type: str
Missing values: 3680
Unique values: 25
Value counts:


Purpose.of.flight
Personal                     11714
NaN                           3680
Instructional                 2753
Aerial Application             935
Business                       499
Unknown                        378
Positioning                    350
Skydiving                      164
Aerial Observation             161
Other Work Use                 155
Flight Test                    120
Ferry                          103
Executive/corporate            103
Banner Tow                      89
Public Aircraft - Federal       52
Air Race show                   48
Public Aircraft                 39
Glider Tow                      35
Public Aircraft - State         24
Firefighting                    17
Public Aircraft - Local         12
ASHO                             5
Air Race/show                    4
Air Drop                         3
PUBS                             3
External Load                    1
Name: count, dtype: int64


Column: Broad.phase.of.flight
Data type: str
Missing values: 18619
Unique values: 12
Value counts:


Broad.phase.of.flight
NaN            18619
Landing         1255
Takeoff          509
Cruise           266
Approach         234
Maneuvering      169
Taxi             111
Go-around         89
Descent           73
Climb             60
Standing          47
Unknown           12
Other              3
Name: count, dtype: int64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [23]:
# Inspect missing values in the current analysis DataFrame
missing_summary = pd.DataFrame({
    "Missing.Count": injury_df.isna().sum(),
    "Missing.Percent": (injury_df.isna().mean() * 100).round(2),
    "Data.Type": injury_df.dtypes.astype(str)
})

missing_summary = missing_summary.sort_values(
    "Missing.Percent",
    ascending=False
)

display(missing_summary)

# Flag columns with at least 70% missing values
missing_threshold = 70

columns_to_review = missing_summary.loc[
    missing_summary["Missing.Percent"] >= missing_threshold
]

print(f"\nColumns with at least {missing_threshold}% missing values:")
display(columns_to_review)

print("\nCurrent DataFrame shape:", injury_df.shape)

,Missing.Count,Missing.Percent,Data.Type
Schedule,18926,88.25,str
Broad.phase.of.flight,18619,86.81,str
Air.carrier,11285,52.62,str
Airport.Code,7464,34.80,str
Airport.Name,7377,34.40,str
Report.Status,4663,21.74,str
Engine.Type,3956,18.45,str
Fatal.Serious.Injury.Fraction,3885,18.11,float64
Purpose.of.flight,3680,17.16,str
Estimated.Occupants,3028,14.12,float64



Columns with at least 70% missing values:


,Missing.Count,Missing.Percent,Data.Type
Schedule,18926,88.25,str
Broad.phase.of.flight,18619,86.81,str



Current DataFrame shape: (21447, 35)


In [24]:
# Record the original shape
original_shape = injury_df.shape

# Drop Schedule because it has excessive missing values
injury_df = injury_df.drop(columns=["Schedule"])

# Verify the change
print("Original DataFrame shape:", original_shape)
print("Updated DataFrame shape:", injury_df.shape)
print("\nSchedule column removed:", "Schedule" not in injury_df.columns)

Original DataFrame shape: (21447, 35)
Updated DataFrame shape: (21447, 34)

Schedule column removed: True


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [25]:
# Save the cleaned DataFrame to the data directory
output_path = "data/AviationData_Cleaned.csv"

injury_df.to_csv(output_path, index=False)

print("Cleaned dataset saved to:", output_path)
print("Dataset shape:", injury_df.shape)

Cleaned dataset saved to: data/AviationData_Cleaned.csv
Dataset shape: (21447, 34)


In [26]:
# Reload the saved CSV to verify it was written successfully
cleaned_check = pd.read_csv(output_path, low_memory=False)

print("Saved file loaded successfully.")
print("Reloaded dataset shape:", cleaned_check.shape)
print("\nFirst five rows:")
display(cleaned_check.head())

print("\nColumns in saved dataset:")
print(cleaned_check.columns.tolist())

Saved file loaded successfully.
Reloaded dataset shape: (21447, 34)

First five rows:


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Estimated.Occupants,Fatal.Serious.Injuries,Fatal.Serious.Injury.Fraction,Aircraft.Destroyed
0,20001214X42478,Incident,LAX83IA149B,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,NaN,588.0,VMC,Standing,Probable Cause,04-12-2014,NaN,NaN,NaN,0.0
1,20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,NaN,588.0,VMC,Taxi,Probable Cause,04-12-2014,NaN,NaN,NaN,0.0
2,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,...,NaN,NaN,IMC,Cruise,Probable Cause,02-05-2011,NaN,2.0,NaN,1.0
3,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,...,NaN,4.0,VMC,Standing,Probable Cause,17-10-2016,NaN,NaN,NaN,NaN
4,20001214X44248,Incident,MIA83IA210,1983-08-21,"NORFOLK, VA",United States,NaN,NaN,NaN,NaN,...,NaN,289.0,VMC,Cruise,Probable Cause,01-02-2016,NaN,NaN,NaN,0.0



Columns in saved dataset:
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date', 'Estimated.Occupants', 'Fatal.Serious.Injuries', 'Fatal.Serious.Injury.Fraction', 'Aircraft.Destroyed']
